In [ ]:
import os
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from google.colab import drive

drive.mount('/content/drive')

# Paths to datasets
DATA_PATHS = {
    "FAO": "/content/drive/MyDrive/FarmWise/Cleaned_Data/FAO/",
    "SoilGrids250m": "/content/drive/MyDrive/FarmWise/Cleaned_Data/SoilGrids250m/",
    "Agridata": "/content/drive/MyDrive/FarmWise/Cleaned_Data/agridata/"
}

# Initialize storage for processed data
processed_data = {}

# ---------------------- PROCESS ALL CSV FILES IN ALL FOLDERS ----------------------
all_csv_data = {}

for key, path in DATA_PATHS.items():
    for root, _, files in os.walk(path):
        csv_files = [f for f in files if f.endswith(".csv")]
        for file in csv_files:
            file_path = os.path.join(root, file)

            # Load CSV file
            df = pd.read_csv(file_path)

            # Drop duplicates
            df.drop_duplicates(inplace=True)

            # Handle missing values: forward fill, then backward fill if needed
            df.fillna(method='ffill', inplace=True)
            df.fillna(method='bfill', inplace=True)

            # Standardize numerical columns if any exist
            num_cols = df.select_dtypes(include=['float64', 'int64']).columns
            if len(num_cols) > 0:
                scaler = StandardScaler()
                df[num_cols] = scaler.fit_transform(df[num_cols])

            all_csv_data[file] = df

processed_data['All_CSV_Data'] = all_csv_data
print("All CSV Data Processed ✔")

# ---------------------- SOILGRIDS250m GEO TIFF PROCESSING ----------------------
soilgrids_path = DATA_PATHS["SoilGrids250m"]
soil_files = [f for f in os.listdir(soilgrids_path) if f.endswith(".tif")]
soil_data = {}

for file in soil_files:
    file_path = os.path.join(soilgrids_path, file)
    with rasterio.open(file_path) as src:
        soil_array = src.read(1)  # Read first band
        soil_array = np.nan_to_num(soil_array)  # Replace NaNs with 0
        soil_data[file] = soil_array

processed_data['SoilGrids250m_GeoTIFF'] = soil_data
print("SoilGrids250m GeoTIFF Data Processed ✔")

# ---------------------- DATA VALIDATION & VISUALIZATION ----------------------
for category, data in processed_data.items():
    print(f"\n--- {category} Data Overview ---")
    if isinstance(data, dict):
        for file, df in data.items():
            print(f"File: {file}")
            print(df.describe())
            print("\n")

            # Plot distributions
            num_cols = df.select_dtypes(include=['float64', 'int64']).columns
            if len(num_cols) > 0:
                df[num_cols].hist(figsize=(10, 5))
                plt.suptitle(f"{file} - Numerical Features Distribution")
                plt.show()

print("✅ All datasets processed, validated, and saved!")
